# Knowledge Distillation on the Beans Dataset - Training Notebook
This notebook loads the dataset, applies preprocessing using the teacher's processor, defines the custom Knowledge Distillation Trainer, and trains a lightweight **MobileNetV2** student model for **10 epochs** using a fine-tuned **ViT-base** teacher model.

In [ ]:
# Environment Setup & Data Loading
!pip install transformers datasets accelerate tensorboard evaluate --upgrade

import torch
from datasets import load_dataset
from transformers import AutoImageProcessor

# Load the beans dataset
dataset = load_dataset("AI-Lab-Makerere/beans")

# Load the public pre-trained teacher's processor
teacher_processor = AutoImageProcessor.from_pretrained("merve/beans-vit-224")

def process(examples):
    processed_inputs = teacher_processor(examples["image"])
    return processed_inputs

# Map preprocessing across all splits (Strictly keeping train/validation/test separate)
processed_datasets = dataset.map(process, batched=True)

In [ ]:
# Define Custom Knowledge Distillation Trainer
from transformers import Trainer, TrainingArguments
import torch.nn as nn
import torch.nn.functional as F

class ImageDistilTrainer(Trainer):
    def __init__(self, teacher_model=None, student_model=None, temperature=None, lambda_param=None, *args, **kwargs):
        super().__init__(model=student_model, *args, **kwargs)
        self.teacher = teacher_model
        self.student = student_model
        self.loss_function = nn.KLDivLoss(reduction="batchmean")
        
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.teacher.to(device)
        self.teacher.eval()
        
        self.temperature = temperature
        self.lambda_param = lambda_param

    def compute_loss(self, student, inputs, return_outputs=False, **kwargs):
        student_output = self.student(**inputs)
        
        with torch.no_grad():
            teacher_output = self.teacher(**inputs)
        
        soft_teacher = F.softmax(teacher_output.logits / self.temperature, dim=-1)
        soft_student = F.log_softmax(student_output.logits / self.temperature, dim=-1)
        
        distillation_loss = self.loss_function(soft_student, soft_teacher) * (self.temperature ** 2)
        student_target_loss = student_output.loss
        
        loss = (1. - self.lambda_param) * student_target_loss + self.lambda_param * distillation_loss
        return (loss, student_output) if return_outputs else loss

In [ ]:
# Models Setup & Initialization (10 Epochs)
from transformers import AutoModelForImageClassification, MobileNetV2Config, MobileNetV2ForImageClassification

training_args = TrainingArguments(
    output_dir="distilled-mobilenet-beans",
    num_train_epochs=10,             # Set to 10 epochs
    fp16=True,                       
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
    push_to_hub=False      
)

num_labels = len(processed_datasets["train"].features["labels"].names)

# Load true fine-tuned teacher model (High accuracy baseline)
teacher_model = AutoModelForImageClassification.from_pretrained(
    "merve/beans-vit-224",
    num_labels=num_labels,
    ignore_mismatched_sizes=False
)

# Initialize blank student architecture
student_config = MobileNetV2Config()
student_config.num_labels = num_labels
student_model = MobileNetV2ForImageClassification(student_config)

In [ ]:
# Evaluation Tools & Data Collators
import evaluate
import numpy as np
from transformers import DefaultDataCollator

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    acc = accuracy.compute(references=labels, predictions=np.argmax(predictions, axis=1))
    return {"accuracy": acc["accuracy"]}

data_collator = DefaultDataCollator()

In [ ]:
# Run Training Loop & Save Model Weights
trainer = ImageDistilTrainer(
    student_model=student_model,
    teacher_model=teacher_model,
    args=training_args,
    train_dataset=processed_datasets["train"],       # Purely training images
    eval_dataset=processed_datasets["validation"],   # Purely validation images
    data_collator=data_collator,
    processing_class=teacher_processor,  
    compute_metrics=compute_metrics,
    temperature=5,
    lambda_param=0.5
)

print("Starting Knowledge Distillation Loop...")
trainer.train()

# Export final model and config so testing notebook can access it without overlap
print("Saving distilled student model...")
trainer.save_model("./saved_distilled_student")
print("Training Complete. Proceed to the Testing Notebook.")